# N1 · 把研究产物装配成论文骨架 + 叙事链审计 (Paper Skeleton Assembler)

> 配套 9.7-L1 · **真实科研动作**: 把你 9.1-9.6 攒下的产物 (领域地图/gap/假设/实验/图) 装配成
> 论文骨架, 并跑**叙事链审计** —— 在动笔前查「哪节缺素材、哪个 claim 没证据」。

写作细则全在 `how_to_write_a_paper` 技能包; 这里只做装配 + 证据链自检。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import paper_assembler as pa
print("标准骨架节:", [name for _, name, _ in pa.SECTIONS])

标准骨架节: ['标题', '摘要', '引言', '相关工作', '方法', '实验', '消融', '讨论', '局限']


## 1. 把你的研究产物登记成字典

以「Robust-DPO 噪声鲁棒性」为例 (对应你的 dpo-family 复现 + 一个 gap)。
**故意漏掉 limitations**, 看审计能不能抓出来。

In [2]:
artifacts = {
    "narrative": "Robust-DPO 在噪声偏好下比 DPO 更鲁棒 (噪声越大优势越明显)",
    "mini_survey": "偏好优化领域地图 (9.2): RLHF→DPO家族→在线化",
    "gap": "DPO 假设偏好标签无噪声 (9.3 假设gap)",
    "hypothesis": "40%噪声下 Robust-DPO win-rate ≥ DPO + 5点 (9.4-L1)",
    "results": "noise=0.4: +13点, p<0.001, 8种子 (9.4/9.5)",
    "ablation": "method×noise 全因子, 交互效应显著 (9.4-L4)",
    "figures": "交互效应图 + Robust-DPO Figure1 (9.6)",
    # "limitations": 故意不填!
}
print("已登记产物:", list(artifacts.keys()))

已登记产物: ['narrative', 'mini_survey', 'gap', 'hypothesis', 'results', 'ablation', 'figures']


## 2. 装配成论文骨架 (每节标出可用素材 / 缺口)

In [3]:
skeleton = pa.assemble_skeleton(artifacts)
out = Path.cwd() / "_paper_output"; out.mkdir(exist_ok=True)
(out / "skeleton.md").write_text(skeleton, encoding="utf-8")
print(skeleton)

# 论文骨架 (auto-assembled)

> 由 9.1-9.6 产物装配. 写作细则见 `how_to_write_a_paper` 技能包 (叙事先行).

## 标题
<!-- 一句话点出贡献, 不卖关子 -->
可用素材: narrative

## 摘要
<!-- 问题→方法→关键结果→意义, 4-6 句 -->
可用素材: narrative, results

## 引言
<!-- 动机 + gap + 我们做了什么 + 贡献列表 -->
可用素材: narrative, mini_survey, gap, hypothesis

## 相关工作
<!-- 领域地图 (9.2), 指出我们补的洞 -->
可用素材: mini_survey, gap

## 方法
<!-- 可证伪假设的方法实现 + Figure 1 示意图 (9.6) -->
可用素材: hypothesis, figures

## 实验
<!-- 主结果 + 公平 baseline (9.4-L3) + 带误差棒图 (9.6) -->
可用素材: results, figures

## 消融
<!-- 消融矩阵 + 交互效应 (9.4-L4) -->
可用素材: ablation

## 讨论
<!-- 为什么 work + 适用边界 -->
⚠ 暂无对应素材 —— 需补 (见 narrative_audit)

## 局限
<!-- 诚实写出, 含不可复现点 (9.5) -->
⚠ 暂无对应素材 —— 需补 (见 narrative_audit)



## 3. 叙事链审计: 动笔前查证据链 (本 notebook 的灵魂)

给出你论文想讲的核心 claim 及其证据来源。审计会抓出: ① 缺素材的节; ② 无证据的 claim。
**故意放一个过度宣称「在所有规模上都成立」, 看它被抓出。**

In [4]:
claims = [
    {"claim": "Robust-DPO 在高噪声下显著更优", "evidence": "results"},
    {"claim": "提升来自鲁棒机制而非工程 trick", "evidence": "ablation"},
    {"claim": "我们的方法在所有模型规模上都成立", "evidence": None},  # ← 过度宣称, 无证据!
]
report = pa.narrative_audit(artifacts, claims)
print(pa.render_audit(report))

叙事链审计: 7/9 节有素材
  ⚠ 缺素材的节: 讨论, 局限
  claim → evidence:
    ✅ 「Robust-DPO 在高噪声下显著更优」← results
    ✅ 「提升来自鲁棒机制而非工程 trick」← ablation
    ❌ 「我们的方法在所有模型规模上都成立」← (无证据!)
  裁决: ⚠ 还有缺口, 补齐再写 (无证据的 claim 是审稿人攻击点)


## 4. 读审计结果: 两个真实的研究决策

审计抓出了两件事, 各对应一个动作:

1. **缺「讨论 / 局限」节** → 补上。Limitations 很多会强制 (9.7-L2), 缺了直接拒。
   你 9.5 的不可复现点、9.4 的适用边界正好填这里。
2. **「在所有规模上都成立」无证据** → 这是审稿人 (尤其 AI 审稿) 必抓的攻击点 (9.7-L1)。
   两个选择: ① 回 9.4 补一个跨规模实验把它坐实; ② 删掉/弱化它 (它属 future work)。
   **笃定地写 ≠ 无证据地吹。**

In [5]:
print("审计裁决: 可动笔吗?", report["ready"])
print("→ 行动: 补 limitations 节; 删除或补证'所有规模'的 claim。")
print("→ 这就是叙事链审计的价值: 在动笔前 (而非被审稿人) 抓出证据链的洞。")

审计裁决: 可动笔吗? False
→ 行动: 补 limitations 节; 删除或补证'所有规模'的 claim。
→ 这就是叙事链审计的价值: 在动笔前 (而非被审稿人) 抓出证据链的洞。


## 5. 反思

你刚把零散研究产物装成了论文骨架, 并在动笔前堵住了两个证据链漏洞。带走:
- 论文 = 用叙事把 9.1-9.6 产物串成**闭合证据链**; 写细则查 `how_to_write_a_paper` 技能包。
- **无证据的 claim 是头号攻击点**; 用 audit 提前抓 (删掉或补实验)。
- Limitations 别漏 (很多会强制)。

下一步: 假设论文投出去了, 收到审稿意见 —— 去 N2 学怎么写 rebuttal。